## 환경 변수 파일 생성

In [3]:
import os
import pathlib

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day01" else here

print(here)
print(ROOT)

os.chdir(ROOT) # 모든 상대경로는 이 폴더가 기준이 된다.

c:\workspace\hanwha-agent\sandbox\w2\day01
c:\workspace\hanwha-agent


* 일반적으로 env 파일을 불러오는 방식

In [4]:
# .env 파일에 적힌 내용을 파이썬 파일(여기)로 가져오기
# dotenv 라이브러리 설치 : python -m pip install python-dotenv
from dotenv import load_dotenv
import os

# .env를 찾아 읽고, os.environ 에 채워 넣는다.
# override=True : 이미 있는 환경변수도 .env 값으로 덮어쓴다. 
# load_dotenv(override=True)
load_dotenv()

print("APP_MODE: ", os.environ.get("APP_MODE"))
print("ANTHROPIC_API_KEY: ", os.environ.get("ANTHROPIC_API_KEY"))
print("MAX_TOKEN: ", os.environ.get("MAX_TOKEN"))
print("DAILY_CALL_LIMIT: ", os.environ.get("DAILY_CALL_LIMIT"))


APP_MODE:  mock
ANTHROPIC_API_KEY:  your-claude-api-key
MAX_TOKEN:  None
DAILY_CALL_LIMIT:  200


* Pydantic을 활용하여 env 정보를 가져오는 방식

In [5]:
# pydantic-settings : 설치 필요
# python -m pip install pydantic-settings

from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field, SecretStr
from functools import lru_cache # 함수 결과를 기억해두는 데코레이터

class Settings(BaseSettings):
    # 앱 설정 : 환경변수와 .env에서 값을 읽어오는 처리
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore", # .env에 모르는 항목이 있어도 무시해라
    )

    # 필드 
    # 필드명 = 환경변수 이름
    # 대소문자 구분하지 않는다.
    app_mode: str = "dev"
    # 기본값 지정이 안되어 있으면 필수값 
    anthropic_api_key: SecretStr # print나 log에 값이 찍히지 않게 변경
    llm_model: str = "claude-haiku-4-5"

    # 타입이 int이므로 "400"을 자동으로 400으로 변환해준다.
    # 직접 int() 형변환을 해주지 않아도 된다.
    max_tokens: int = Field(default=400, ge=1, le=8192)
    daily_call_limit: int = Field(default=200, ge=1)
    max_input_chars: int = Field(default=200, ge=1)

# settings = Settings()

# print(f"app_mode: {settings.app_mode}")
# print(f"llm_model: {settings.llm_model}")
# print(f"max_tokens: {settings.max_tokens}")
# print(f"anthropic_api_key: {settings.anthropic_api_key}")
# print(f"daily_call_limit: {settings.daily_call_limit}")

# 이 함수는 처음 한 번만 실제로 실행
@lru_cache 
def get_settings() -> Settings:
    print("Settings 객체 생성")
    return Settings()

print("첫 번째 호출")
s1 = get_settings()
print("두 번쨰 호출")
s2 = get_settings()
print()
print(f"같은 객체인가? : {s1} is {s2}")

첫 번째 호출
Settings 객체 생성
두 번쨰 호출

같은 객체인가? : app_mode='mock' anthropic_api_key=SecretStr('**********') llm_model='claude-haiku-4-5' max_tokens=400 daily_call_limit=200 max_input_chars=200 is app_mode='mock' anthropic_api_key=SecretStr('**********') llm_model='claude-haiku-4-5' max_tokens=400 daily_call_limit=200 max_input_chars=200
